# Evaluate Flagged Output Against Reviewer Gold Set

Loads the reviewer-adjudicated spreadsheet (`Rivas_Ture_combinedreview_flaggedterms.xlsx`) as the gold standard and scores a candidate flagged-output CSV against it.

**Gold-label conventions** (from the reviewer workflow):
- `false_positive_term == 'x'` -> the model-flagged term is a **false positive**.
- otherwise (blank) -> the model-flagged term is a **true positive** (reviewer-confirmed bias).
- `missed_term` populated -> reviewer-identified **false negative** at the listed term/context.

**Outputs**
1. Overall precision (TP / TP+FP) and recall vs. missed terms.
2. Per-category precision.
3. Per-(normalized_term, category) FP rate -- to retire individual term-bank entries.
4. Dax-vs-Human FP-rate parity (key confounder check).

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

GOLD_PATH = Path("PUT_PATH_TO_GOLD_XLSX_HERE.xlsx")

# Set this to a flagged-output CSV produced by bias_pipeline (e.g., output/bias_flagged_*.csv).
# Leave as None to run sanity check against the gold set itself.
CANDIDATE_CSV: Path | None = Path("PUT_PATH_TO_CANDIDATE_CSV_HERE.csv")

## 1. Load gold set

In [ ]:
gold = pd.read_excel(GOLD_PATH)
gold["is_fp"] = gold["false_positive_term"].astype(str).str.strip().str.lower() == "x"
gold["normalized_term"] = gold["normalized_term"].astype(str).str.lower().str.strip()
gold["model_category"] = gold["model_category"].astype(str).str.lower().str.strip()
n_total = len(gold)
n_fp = int(gold["is_fp"].sum())
n_tp = int((~gold["is_fp"]).sum())
print(f"Gold rows: {n_total} | FPs: {n_fp} | TPs: {n_tp}")
print(f"Notes covered: {gold['source_note_id'].nunique()}")
missed = gold[gold["missed_term"].notna()][
    ["source_note_id", "missed_term", "term_context"]
].drop_duplicates()
print(f"Missed-term annotations: {len(missed)}")
missed.head()

## 2. Load candidate flagged-output CSV

The candidate CSV is the output of `bias_pipeline` (timestamped `bias_flagged_*.csv` in `output/`). It is exploded into one row per flagged term via `build_reviewer_adjudication_dataframe` so its shape matches the gold set.

In [ ]:
from bias_pipeline import build_reviewer_adjudication_dataframe

if CANDIDATE_CSV is None:
    candidate = gold.copy()
    print("CANDIDATE_CSV is None -- running sanity check against gold set itself.")
else:
    raw = pd.read_csv(CANDIDATE_CSV)
    candidate = build_reviewer_adjudication_dataframe(raw)
    candidate["normalized_term"] = (
        candidate["normalized_term"].astype(str).str.lower().str.strip()
    )
    candidate["model_category"] = (
        candidate["model_category"].astype(str).str.lower().str.strip()
    )
    print(f"Candidate flagged rows: {len(candidate)} from {CANDIDATE_CSV.name}")
    print(
        f"Candidate prompt versions: {sorted(candidate['Prompt_Version'].dropna().unique())}"
    )
    print(f"Candidate models: {sorted(candidate['Model_Used'].dropna().unique())}")

## 3. Match candidate rows to gold labels

Match key: `(source_note_id, normalized_term, model_category)`. A candidate row is **scored** if a matching gold row exists; **unscored** rows fall outside the reviewed sample and are reported separately.

In [ ]:
gold_key = gold[
    ["source_note_id", "normalized_term", "model_category", "is_fp", "reviewer_notes"]
].copy()
gold_key = gold_key.drop_duplicates(
    subset=["source_note_id", "normalized_term", "model_category"], keep="first"
)

key_cols = ["source_note_id", "normalized_term", "model_category"]
scored = candidate.merge(gold_key, on=key_cols, how="left", suffixes=("", "_gold"))
scored["scored"] = scored["is_fp"].notna()
print(
    f"Candidate rows matched to a gold label: {int(scored['scored'].sum())} / {len(scored)}"
)
print(f"Unmatched (outside reviewed sample): {int((~scored['scored']).sum())}")

## 4. Overall precision and FP rate

In [ ]:
matched = scored[scored["scored"]].copy()
matched["is_fp"] = matched["is_fp"].astype(
    bool
)  # merge can upcast bool->object; coerce back
tp = int((~matched["is_fp"]).sum())
fp = int(matched["is_fp"].sum())
precision = tp / (tp + fp) if (tp + fp) else float("nan")
fp_rate = fp / (tp + fp) if (tp + fp) else float("nan")
print(f"TP: {tp}   FP: {fp}   Precision: {precision:.3f}   FP rate: {fp_rate:.3f}")

## 5. Per-category precision

In [ ]:
per_cat = (
    matched.groupby("model_category")["is_fp"]
    .agg(n="count", fp="sum")
    .assign(tp=lambda d: d["n"] - d["fp"], precision=lambda d: 1 - d["fp"] / d["n"])
    .sort_values("n", ascending=False)
)
per_cat

## 6. Per-term FP rate (term-bank pruning candidates)

Terms with high FP rate and a meaningful sample size are candidates for removal from `TERM_BANK` or for tightening exception rules in `bias_detection_prompt.py`.

In [ ]:
per_term = (
    matched.groupby(["normalized_term", "model_category"])["is_fp"]
    .agg(n="count", fp="sum")
    .assign(fp_rate=lambda d: d["fp"] / d["n"])
    .sort_values(["fp_rate", "n"], ascending=[False, False])
)
print("Top FP-rate term/category pairs (n >= 2):")
per_term[per_term["n"] >= 2].head(30)

## 7. Dax vs Human FP-rate parity

If structured DAX templates (SDOH screening blocks, screening questionnaires) inflate FPs disproportionately, the H-vs-DAX bias comparison is confounded. Target: parity within ~5 percentage points.

In [ ]:
by_source = (
    matched.groupby("Dax_or_Human")["is_fp"]
    .agg(n="count", fp="sum")
    .assign(fp_rate=lambda d: d["fp"] / d["n"])
)
by_source

## 8. Recall on reviewer-identified missed terms

Recall = (missed terms now flagged by the candidate) / (total missed-term annotations). Only meaningful once `missed_term` is populated in the gold set.

In [ ]:
missed_terms = gold[gold["missed_term"].notna()][
    ["source_note_id", "missed_term"]
].copy()
missed_terms["missed_term"] = (
    missed_terms["missed_term"].astype(str).str.lower().str.strip()
)
missed_terms = missed_terms.drop_duplicates()

candidate_terms_by_note = (
    candidate.assign(t=candidate["normalized_term"])
    .groupby("source_note_id")["t"]
    .apply(lambda s: set(s.tolist()))
)


def caught(row):
    terms = candidate_terms_by_note.get(row["source_note_id"], set())
    return any(row["missed_term"] in t or t in row["missed_term"] for t in terms)


if len(missed_terms):
    missed_terms["caught"] = missed_terms.apply(caught, axis=1)
    recall = float(missed_terms["caught"].mean())
    print(
        f"Missed-term annotations: {len(missed_terms)}   Caught: {int(missed_terms['caught'].sum())}   Recall: {recall:.3f}"
    )
    print()
    print(missed_terms.to_string(index=False))
else:
    print("No missed-term annotations in gold set.")

## 9. Summary line for the run log

Drop this single line into a run-log to track prompt-version progress.

In [ ]:
prompt_versions = (
    sorted(candidate["Prompt_Version"].dropna().unique().tolist())
    if "Prompt_Version" in candidate.columns
    else []
)
model_used = (
    sorted(candidate["Model_Used"].dropna().unique().tolist())
    if "Model_Used" in candidate.columns
    else []
)
summary = {
    "prompt_version": prompt_versions,
    "model": model_used,
    "matched_rows": int(matched.shape[0]),
    "tp": tp,
    "fp": fp,
    "precision": round(precision, 3) if precision == precision else None,
    "dax_fp_rate": round(float(by_source.loc["Dax", "fp_rate"]), 3)
    if "Dax" in by_source.index
    else None,
    "human_fp_rate": round(float(by_source.loc["Human", "fp_rate"]), 3)
    if "Human" in by_source.index
    else None,
}
summary